In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install datasets torch pandas numpy scikit-learn tqdm

Mounted at /content/drive


In [3]:
from datasets import load_dataset

ds = load_dataset("sneakyzero/cs2-pro-round-economy", split="train")
print(ds.column_names)
print(ds[0])

README.md:   0%|          | 0.00/6.61k [00:00<?, ?B/s]

data/full_player_round_economy.parquet: reconstructing file:   0%|          |  0.00B / 6.62MB            

data/full_player_round_economy.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

['map_uid', 'match_id', 'map_index', 'event', 'team1', 'team2', 'series_score1', 'series_score2', 'map_name', 'match_date', 'patch_version', 'source_file_name', 'source_repo', 'source_revision', 'parser_name', 'parser_version', 'schema_version', 'extracted_at_utc', 'round_num', 'freeze_end_tick', 'steamid', 'player_name', 'team_num', 'side', 'team_name', 'team_clan_name', 'team_name_resolved', 'team_name_resolution_method', 'roster_uid', 'roster_index_in_demo_metadata', 'roster_side_start', 'balance', 'start_balance', 'cash_spent_this_round', 'armor_value', 'has_helmet', 'has_defuser', 'round_start_equip_value', 'current_equip_value', 'inventory_json', 'is_connected', 'player_active_at_freeze_end']
{'map_uid': '2392432-1', 'match_id': '2392432', 'map_index': 1, 'event': 'PGL Astana 2026 Europe Open Qualifier 1', 'team1': 'magic', 'team2': 'Inner Circle Academy', 'series_score1': 2, 'series_score2': 0, 'map_name': 'de_overpass', 'match_date': datetime.datetime(2026, 3, 23, 11, 4, 9, tzi

In [4]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Load the dataset
ds = load_dataset("sneakyzero/cs2-pro-round-economy", split="train")
df = pd.DataFrame(ds)

# Check available tables/features
print("Columns:", df.columns.tolist())
print("Sample:", df.iloc[0])

# Aggregate to team level per round
team_round_agg = df.groupby(['map_uid', 'round_num', 'side']).agg({
    'balance': 'sum',
    'current_equip_value': 'sum',
    'has_helmet': 'sum',
    'has_defuser': 'sum',
    'armor_value': lambda x: (x > 0).sum(),  # count players with armor
    'player_active_at_freeze_end': 'sum'  # count active players
}).reset_index()

team_round_agg.columns = ['map_uid', 'round_num', 'side', 'team_cash', 'team_equip', 'helmet_count', 'defuser_count', 'armor_count', 'active_players']

print("\nTeam aggregates:")
print(team_round_agg.head())

# Pivot to get T and CT on same row
pivoted = team_round_agg.pivot_table(
    index=['map_uid', 'round_num'],
    columns='side',
    values=['team_cash', 'team_equip', 'helmet_count', 'defuser_count', 'armor_count'],
    aggfunc='first'
).reset_index()

pivoted.columns = ['_'.join(col).strip('_') for col in pivoted.columns.values]
print("\nPivoted:")
print(pivoted.head())

Columns: ['map_uid', 'match_id', 'map_index', 'event', 'team1', 'team2', 'series_score1', 'series_score2', 'map_name', 'match_date', 'patch_version', 'source_file_name', 'source_repo', 'source_revision', 'parser_name', 'parser_version', 'schema_version', 'extracted_at_utc', 'round_num', 'freeze_end_tick', 'steamid', 'player_name', 'team_num', 'side', 'team_name', 'team_clan_name', 'team_name_resolved', 'team_name_resolution_method', 'roster_uid', 'roster_index_in_demo_metadata', 'roster_side_start', 'balance', 'start_balance', 'cash_spent_this_round', 'armor_value', 'has_helmet', 'has_defuser', 'round_start_equip_value', 'current_equip_value', 'inventory_json', 'is_connected', 'player_active_at_freeze_end']
Sample: map_uid                                                                  2392432-1
match_id                                                                   2392432
map_index                                                                        1
event                     

In [5]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

# Load both splits
ds_economy = load_dataset("sneakyzero/cs2-pro-round-economy", split="train")
ds_rounds = load_dataset("sneakyzero/cs2-pro-round-economy", name="rounds", split="train")

df_economy = pd.DataFrame(ds_economy)
df_rounds = pd.DataFrame(ds_rounds)

# Aggregate economy to team level
team_round_agg = df_economy.groupby(['map_uid', 'round_num', 'side']).agg({
    'balance': 'sum',
    'current_equip_value': 'sum',
    'has_helmet': 'sum',
    'has_defuser': 'sum',
    'armor_value': lambda x: (x > 0).sum(),
    'player_active_at_freeze_end': 'sum'
}).reset_index()

team_round_agg.columns = ['map_uid', 'round_num', 'side', 'team_cash', 'team_equip', 'helmet_count', 'defuser_count', 'armor_count', 'active_players']

# Pivot to get T and CT on same row
pivoted = team_round_agg.pivot_table(
    index=['map_uid', 'round_num'],
    columns='side',
    values=['team_cash', 'team_equip', 'helmet_count', 'defuser_count', 'armor_count'],
).reset_index()

pivoted.columns = ['_'.join(col).strip('_') if col[1] else col[0] for col in pivoted.columns.values]

# Merge with round outcomes
rounds_slim = df_rounds[['map_uid', 'round_num', 'winner_side']]
merged = pivoted.merge(rounds_slim, on=['map_uid', 'round_num'])

# Create features: [t_cash, t_equip, t_helmet, t_defuser, t_armor, ct_cash, ct_equip, ct_helmet, ct_defuser, ct_armor]
feature_cols = ['team_cash_t', 'team_equip_t', 'helmet_count_t', 'defuser_count_t', 'armor_count_t',
                'team_cash_ct', 'team_equip_ct', 'helmet_count_ct', 'defuser_count_ct', 'armor_count_ct']

X = merged[feature_cols].fillna(0).values.astype(np.float32)
y = (merged['winner_side'] == 't').values.astype(np.int64)  # 1 if T wins, 0 if CT wins

# Normalize features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Create tensors
X_tensor = torch.from_numpy(X).float()
y_tensor = torch.from_numpy(y).long()

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Features shape: {X.shape}")
print(f"Class distribution: {(y==0).sum()} CT wins, {(y==1).sum()} T wins")

data/full_rounds.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

data/full_rounds.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset size: 42009
Features shape: (42009, 10)
Class distribution: 21625 CT wins, 20384 T wins


In [9]:
from sklearn.model_selection import train_test_split

# Split before DataLoader
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [10]:
from xgboost import XGBClassifier

gb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

gb_model.fit(X_train, y_train)
y_pred = gb_model.predict(X_test)

from sklearn.metrics import accuracy_score, roc_auc_score
print(f"XGBoost Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"XGBoost ROC-AUC: {roc_auc_score(y_test, y_pred):.4f}")

XGBoost Test Accuracy: 0.6331
XGBoost ROC-AUC: 0.6316


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

# Model
class RoundOutcomePredictor(nn.Module):
    def __init__(self, input_size=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.mlp(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RoundOutcomePredictor().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# Training
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0

    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device).float().unsqueeze(1)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = (logits > 0).float()
        correct += (pred == y_batch).sum().item()

    accuracy = correct / len(dataset)
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f} Accuracy: {accuracy:.4f}")

# Save
torch.save(model.state_dict(), '/content/drive/MyDrive/round_outcome_model.pt')
print("Model saved to Drive")

Epoch 1: 100%|██████████| 132/132 [00:01<00:00, 116.55it/s]


Epoch 1 Loss: 0.6261 Accuracy: 0.5025


Epoch 2: 100%|██████████| 132/132 [00:00<00:00, 313.63it/s]


Epoch 2 Loss: 0.6084 Accuracy: 0.5096


Epoch 3: 100%|██████████| 132/132 [00:00<00:00, 286.55it/s]


...

Epoch 18 Loss: 0.6025 Accuracy: 0.5127


Epoch 19: 100%|██████████| 132/132 [00:00<00:00, 228.09it/s]


Epoch 19 Loss: 0.6019 Accuracy: 0.5128


Epoch 20: 100%|██████████| 132/132 [00:00<00:00, 208.34it/s]


Epoch 20 Loss: 0.6018 Accuracy: 0.5143
Model saved to Drive


In [12]:
model.eval()
with torch.no_grad():
    all_preds, all_true = [], []
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = model(X_batch)
        preds = (logits > 0).cpu().numpy().flatten()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

print(f"Test Accuracy: {accuracy_score(all_true, all_preds):.4f}")
print(f"Test ROC-AUC: {roc_auc_score(all_true, all_preds):.4f}")

Test Accuracy: 0.6419
Test ROC-AUC: 0.6400


In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd

# Redefine class
class RoundOutcomePredictor(nn.Module):
    def __init__(self, input_size=10):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.mlp(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Feature engineering
X_eng = np.hstack([
    X,
    (X[:, 1] / (X[:, 6] + 1))[:, None],  # T equip / CT equip
    (X[:, 0] - X[:, 5])[:, None],        # T cash - CT cash
    ((X[:, 2] + X[:, 3]) - (X[:, 7] + X[:, 8]))[:, None],  # T utility - CT utility
])

scaler_eng = StandardScaler()
X_eng = scaler_eng.fit_transform(X_eng)

X_train_eng, X_test_eng, _, _ = train_test_split(
    torch.from_numpy(X_eng).float(), y_tensor,
    test_size=0.2, random_state=42, stratify=y_tensor
)

# Train MLP with engineered features
model_eng = RoundOutcomePredictor(input_size=13).to(device)
optimizer_eng = optim.Adam(model_eng.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

train_dataset_eng = TensorDataset(X_train_eng, y_train)
train_loader_eng = DataLoader(train_dataset_eng, batch_size=256, shuffle=True)

for epoch in range(20):
    model_eng.train()
    for X_batch, y_batch in train_loader_eng:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device).float().unsqueeze(1)
        optimizer_eng.zero_grad()
        logits = model_eng(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer_eng.step()

# Evaluate
model_eng.eval()
with torch.no_grad():
    logits_eng = model_eng(X_test_eng.to(device))
    y_pred_eng = (logits_eng > 0).cpu().numpy().flatten()

acc_eng = accuracy_score(y_test.numpy(), y_pred_eng)
auc_eng = roc_auc_score(y_test.numpy(), y_pred_eng)

# Comparison table
results = pd.DataFrame({
    'Model': ['MLP (baseline)', 'MLP + Features', 'XGBoost'],
    'Accuracy': [0.6427, acc_eng, 0.6331],
    'ROC-AUC': [0.6399, auc_eng, 0.6316]
})

print(results.to_string(index=False))
results.to_csv('/content/drive/MyDrive/model_comparison.csv', index=False)

         Model  Accuracy  ROC-AUC
MLP (baseline)  0.642700 0.639900
MLP + Features  0.642704 0.639699
       XGBoost  0.633100 0.631600


In [16]:
# Save final comparison
results.to_csv('/content/drive/MyDrive/model_comparison.csv', index=False)

# Save engineered features model checkpoint
torch.save(model_eng.state_dict(), '/content/drive/MyDrive/round_outcome_model_engineered.pt')